# NB_SEC_Common · shared configuration and helpers
Loaded by the other NB_SEC_* notebooks with `%run NB_SEC_Common`. Values in CONFIG were filled in by fabric_deploy.py.
Every call to Graph, Fabric and SQL runs as the service principal spn-fabric-secops (secret read from Key Vault).

In [ ]:
CONFIG = {
    "tenant_id": "{{TENANT_ID}}",
    "client_id": "{{CLIENT_ID}}",
    "key_vault_url": "{{KEY_VAULT_URL}}",
    "secret_name": "spn-fabric-secops-secret",
    "sql_server": "{{SQL_SERVER}}",          # host only, no port
    "sql_database": "{{SQL_DATABASE}}",
    "mail_sender": "{{MAIL_SENDER}}",        # mailbox that sends notifications
    "security_team_mail": "{{SECURITY_MAIL}}",
    "tenant_domain": "{{TENANT_DOMAIN}}",
}

In [ ]:
import requests, time, json, hashlib, datetime as dt
import pandas as pd
_SECRET = notebookutils.credentials.getSecret(CONFIG["key_vault_url"], CONFIG["secret_name"])
_TOK = {}
FABRIC = "https://api.fabric.microsoft.com/v1"
GRAPH = "https://graph.microsoft.com/v1.0"
SCOPE = {"fabric": "https://api.fabric.microsoft.com/.default", "graph": "https://graph.microsoft.com/.default", "sql": "https://database.windows.net/.default"}

def token(kind):
    t = _TOK.get(kind)
    if t and t[1] > time.time() + 300:
        return t[0]
    r = requests.post(f"https://login.microsoftonline.com/{CONFIG['tenant_id']}/oauth2/v2.0/token",
                      data={"grant_type": "client_credentials", "client_id": CONFIG["client_id"], "client_secret": _SECRET, "scope": SCOPE[kind]})
    r.raise_for_status(); j = r.json()
    _TOK[kind] = (j["access_token"], time.time() + int(j["expires_in"]))
    return j["access_token"]

def call(kind, method, url, ok=(200, 201, 202, 204), **kw):
    base = FABRIC if kind == "fabric" else GRAPH
    for attempt in range(6):
        r = requests.request(method, url if url.startswith("http") else base + url,
                             headers={"Authorization": f"Bearer {token(kind)}", **kw.pop("headers", {})}, **kw)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(int(r.headers.get("Retry-After", 2 ** attempt))); continue
        if ok and r.status_code not in ok:
            raise RuntimeError(f"{method} {url} -> {r.status_code}: {r.text[:500]}")
        return r
    raise RuntimeError(f"{method} {url} failed after retries")

def fabric_all(path):
    out, url = [], path
    while url:
        j = call("fabric", "GET", url).json(); out += j.get("value", []); url = j.get("continuationUri")
    return out

def graph_all(path):
    out, url = [], path
    while url:
        j = call("graph", "GET", url).json(); out += j.get("value", []); url = j.get("@odata.nextLink")
    return out

In [ ]:
# SQL helpers for SQLDB_SecMeta (JDBC with an Entra access token)
JDBC_URL = (f"jdbc:sqlserver://{CONFIG['sql_server']}:1433;database={{{CONFIG['sql_database']}}};"
            "encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30")

def sql_df(query):
    return (spark.read.format("jdbc").option("url", JDBC_URL).option("query", query)
            .option("accessToken", token("sql")).option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
            .load().toPandas())

def _conn():
    jvm = spark._sc._gateway.jvm
    props = jvm.java.util.Properties(); props.setProperty("accessToken", token("sql"))
    return jvm.java.sql.DriverManager.getConnection(JDBC_URL, props)

def sql_exec(stmt, params=(), fetch_scalar=False):
    con = _conn()
    try:
        ps = con.prepareStatement(stmt)
        for i, v in enumerate(params, start=1):
            if v is None: ps.setNull(i, 12)
            elif isinstance(v, bool): ps.setBoolean(i, v)
            elif isinstance(v, int): ps.setLong(i, v)
            else: ps.setString(i, str(v))
        if fetch_scalar:
            rs = ps.executeQuery(); rs.next(); return rs.getObject(1)
        return ps.executeUpdate()
    finally:
        con.close()

NOW = lambda: dt.datetime.utcnow().replace(microsecond=0)
TODAY = dt.date.today()
ACTOR = "spn-fabric-secops"

def audit(action, target, run_id=None, request_id=None, before=None, after=None, outcome="Success", actor=ACTOR):
    sql_exec("INSERT INTO sec.AuditLog (EventTime,RunId,RequestId,Action,Target,BeforeJson,AfterJson,Actor,Outcome) VALUES (?,?,?,?,?,?,?,?,?)",
             (NOW().isoformat(sep=" "), run_id, request_id, action, target[:200],
              None if before is None else json.dumps(before), None if after is None else json.dumps(after), actor, outcome))

def queue_mail(request_id, ntype, to, cc=None, when=None):
    exists = sql_exec("SELECT COUNT(*) FROM sec.NotificationLog WHERE RequestId = ? AND NotificationType = ?", (request_id, ntype), fetch_scalar=True)
    if ntype in ("EXPIRY_T7", "EXPIRY_T1", "ENABLED", "REVOKED") and exists:
        return False
    sql_exec("INSERT INTO sec.NotificationLog (RequestId,NotificationType,SentTo,Cc,ScheduledFor,Status) VALUES (?,?,?,?,?,'Queued')",
             (request_id, ntype, to, cc, (when or NOW()).isoformat(sep=" ")))
    return True
print("NB_SEC_Common loaded")